# Postprocess into rasters and vector datasets




In [6]:
from pathlib import Path

import geopandas as gpd
import numpy as np
from osgeo import gdal
import rasterstats
from osgeo_utils import gdal_calc


import subkart

In [7]:
nodata = 255
crs = "EPSG:25833"

In [8]:
classifier = subkart.utils.load_classifier()

## Post processing

In [9]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}_unmapped.tif"
prob_file = f"{fname}_3band_probability.tif"

In [10]:
gdal.UseExceptions()

prediction_files = [f"{r}_prediction.tif" for r in subkart.sources.REGIONS]
probability_files = [f"{r}_probability.tif" for r in subkart.sources.REGIONS]

subkart.utils.merge_rasters(prediction_files, predict_file, nodata=nodata)
subkart.utils.merge_rasters(probability_files, prob_file, nodata=-9999)


# Create processed prediction raster:

Remap class 1 (blanding) → highest-probability class (0=løsbunn or 2=fastbunn) using gdal_calc

In [18]:
predict_file_remapped = f"{fname}.tif"

In [19]:

gdal_calc.Calc(
    calc="numpy.where(A==1, numpy.where(B >= C, numpy.uint8(0), numpy.uint8(2)), A)",
    outfile=predict_file_remapped,
    A=predict_file,
    B=prob_file,
    B_band=1,
    C=prob_file,
    C_band=3,
    type="Byte",
    NoDataValue=nodata,
    hideNoData=True,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)



0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:19.


<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7f43e3d9e4c0> >

## Create processed 1-band probability raster from the 3-band source

* class 0 (løsbunn, incl. remapped/sieved blanding) band 1 = P(class=0)
* class 2 (fastbunn)                            band 3 = P(class=2)

In [13]:

prob_file_processed = f"{fname}_probability.tif"

gdal_calc.Calc(
    calc="numpy.where(A==2, C/(B+C), B/(B+C))",
    outfile=prob_file_processed,
    A=predict_file_remapped,
    B=prob_file,
    B_band=1,
    C=prob_file,
    C_band=3,
    type="Float32",
    NoDataValue=-9999,
    creation_options=["COMPRESS=DEFLATE", "TILED=YES", "BLOCKXSIZE=512", "BLOCKYSIZE=512", "BIGTIFF=IF_SAFER"],
    overwrite=True,
)


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:29.


<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7f454f5c5da0> >

## Vectorize processed prediction raster

In [ ]:
subkart.vectorize.with_gdal(
    predict_file_remapped, "polygons_processed.gpkg", epsg_code=int(crs.split(":")[1])
)

gdf = gpd.read_file("polygons_processed.gpkg")
gdf = gdf.dissolve(by="DN", as_index=False)
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-9999,
)
gdf["sannsynlighet"] = [s["mean"]*100 for s in stats]

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")


Polygons saved to polygons_processed.gpkg


In [ ]:
subkart.utils.to_postgis(gdf, fname)